**Student Name:** Lana Ismail

**Student ID:** 20210229

In [ ]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# **Task 1**

The "Bank Marketing Data Set" from the UCI Machine Learning Repository is related with direct marketing campaigns (phone calls) of a Portuguese banking institution.

The classification goal is to predict if the client will subscribe a term deposit (variable y). You can find a description of the attributes at the original UCI URL,
https://archive.ics.uci.edu/ml/datasets/Bank+Marketing

The UCI page contains multiple versions of the data, so the version that you need to work with is on E-Learning.

Your task is to build two classifiers for this data set: a decision tree and a naïve Bayes classifier. Use a suitable evaluation metric to compare the performance of the classifiers.


In [ ]:
#first step: read data and view a subset from it
#used (sep = ;) because of the structure of data provided, it will split it on ";"

df = pd.read_csv("/content/bank-additional-full.csv",sep=';')
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
#check the dimension (shape) of the dataframe
df.shape

(41188, 21)

In [ ]:
#check column names
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [ ]:
#check for missing values to handle if exist
df.isnull().sum()

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [ ]:
#check datatype for each column, to convert any if needed
df.dtypes

age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object

In [ ]:
#check number of unique values in each column, to know what techinque is better to use when converting to categorical
df.nunique()

age                 78
job                 12
marital              4
education            8
default              3
housing              3
loan                 3
contact              2
month               10
day_of_week          5
duration          1544
campaign            42
pdays               27
previous             8
poutcome             3
emp.var.rate        10
cons.price.idx      26
cons.conf.idx       26
euribor3m          316
nr.employed         11
y                    2
dtype: int64

In [ ]:
#drop "duration" feature because it affects the result as mentioned in the description
df.drop('duration', axis = 1, inplace= True)

In [ ]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous',
       'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
       'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [ ]:
df.shape

(41188, 20)

In [ ]:
#check unique values for "education" to see which is better to use when converting it
df['education'].unique()

array(['basic.4y', 'high.school', 'basic.6y', 'basic.9y',
       'professional.course', 'unknown', 'university.degree',
       'illiterate'], dtype=object)

In [ ]:
df['contact'].unique()

array(['telephone', 'cellular'], dtype=object)

In [ ]:
df['job'].unique()

array(['housemaid', 'services', 'admin.', 'blue-collar', 'technician',
       'retired', 'management', 'unemployed', 'self-employed', 'unknown',
       'entrepreneur', 'student'], dtype=object)

In [ ]:
#used one-hot encoder to convert (marital,job,education) from categorical to numerical, because they are not ordinal and using label encoder might affect the result
df = pd.get_dummies(df, columns = ['job','marital','education'])
df.head()

,age,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,...,marital_single,marital_unknown,education_basic.4y,education_basic.6y,education_basic.9y,education_high.school,education_illiterate,education_professional.course,education_university.degree,education_unknown
0,56,no,no,no,telephone,may,mon,1,999,0,...,0,0,1,0,0,0,0,0,0,0
1,57,unknown,no,no,telephone,may,mon,1,999,0,...,0,0,0,0,0,1,0,0,0,0
2,37,no,yes,no,telephone,may,mon,1,999,0,...,0,0,0,0,0,1,0,0,0,0
3,40,no,no,no,telephone,may,mon,1,999,0,...,0,0,0,1,0,0,0,0,0,0
4,56,no,no,yes,telephone,may,mon,1,999,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
#used label encoder to convert these features to numerical
#decided to treat unknown as a possible class label

categorical=['poutcome','default','housing'	,'loan','y']

labelEncoder = LabelEncoder()
for col in categorical:
  df[col] = labelEncoder.fit_transform(df[col])

df.head()

,age,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,...,marital_single,marital_unknown,education_basic.4y,education_basic.6y,education_basic.9y,education_high.school,education_illiterate,education_professional.course,education_university.degree,education_unknown
0,56,0,0,0,telephone,may,mon,1,999,0,...,0,0,1,0,0,0,0,0,0,0
1,57,1,0,0,telephone,may,mon,1,999,0,...,0,0,0,0,0,1,0,0,0,0
2,37,0,2,0,telephone,may,mon,1,999,0,...,0,0,0,0,0,1,0,0,0,0
3,40,0,0,0,telephone,may,mon,1,999,0,...,0,0,0,1,0,0,0,0,0,0
4,56,0,0,2,telephone,may,mon,1,999,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
df.nunique()

age                               78
default                            3
housing                            3
loan                               3
contact                            2
month                             10
day_of_week                        5
campaign                          42
pdays                             27
previous                           8
poutcome                           3
emp.var.rate                      10
cons.price.idx                    26
cons.conf.idx                     26
euribor3m                        316
nr.employed                       11
y                                  2
job_admin.                         2
job_blue-collar                    2
job_entrepreneur                   2
job_housemaid                      2
job_management                     2
job_retired                        2
job_self-employed                  2
job_services                       2
job_student                        2
job_technician                     2
j

In [ ]:
df.dtypes

age                                int64
default                            int64
housing                            int64
loan                               int64
contact                           object
month                             object
day_of_week                       object
campaign                           int64
pdays                              int64
previous                           int64
poutcome                           int64
emp.var.rate                     float64
cons.price.idx                   float64
cons.conf.idx                    float64
euribor3m                        float64
nr.employed                      float64
y                                  int64
job_admin.                         uint8
job_blue-collar                    uint8
job_entrepreneur                   uint8
job_housemaid                      uint8
job_management                     uint8
job_retired                        uint8
job_self-employed                  uint8
job_services    

In [ ]:
df.shape

(41188, 41)

In [ ]:
#splitting the data to X and y, for X i dropped the target (y), (contact,month,day_of_week) because i don't think they will be useful for prediction

X = df.drop(columns = ['y','contact','month','day_of_week'])
y = df['y']

In [ ]:
#split the data to train 70% and test 30%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

**Decision Tree Classifier**

In [ ]:
decisionTree = DecisionTreeClassifier()
decisionTree.fit(X_train,y_train)

y_predict = decisionTree.predict(X_test)
accuracy = accuracy_score(y_test, y_predict)
report = classification_report(y_test, y_predict)

print("Accuracy of decision Tree:", accuracy)
print('\n\t\t\t -REPORT- \n')
print(report)

Accuracy of decision Tree: 0.8421137816622157

			 -REPORT- 

              precision    recall  f1-score   support

           0       0.92      0.91      0.91     10968
           1       0.31      0.34      0.33      1389

    accuracy                           0.84     12357
   macro avg       0.61      0.62      0.62     12357
weighted avg       0.85      0.84      0.84     12357



In [ ]:
# 1- build (Decision Tree Classifier)
# 2- train the classifier (using fit)
# 3- apply what it learned during fitting to predict the class (using predict)
# 4- evaluate the performance of the classifier
# 5- report that includes precision, recall, F1-score, and support for evaluating the performance of the classifier

**Naïve Bayes Classifier**

In [ ]:
naiveBayes = GaussianNB()
naiveBayes.fit(X_train,y_train)

y_predict2 = naiveBayes.predict(X_test)
accuracy2 = accuracy_score(y_test, y_predict2)
report2 = classification_report(y_test, y_predict2)

print("Accuracy of naive Bayes:", accuracy2)
print('\n\t\t\t -REPORT- \n')
print(report2)

Accuracy of naive Bayes: 0.8341830541393542

			 -REPORT- 

              precision    recall  f1-score   support

           0       0.93      0.88      0.90     10968
           1       0.34      0.49      0.40      1389

    accuracy                           0.83     12357
   macro avg       0.63      0.68      0.65     12357
weighted avg       0.86      0.83      0.85     12357



In [ ]:
# 1- build (Naïve Bayes Classifier)
# 2- train the classifier (using fit)
# 3- apply what it learned during fitting to predict the class (using predict)
# 4- evaluate the performance of the classifier
# 5- report that includes precision, recall, F1-score, and support for evaluating the performance of the classifier

**Decision Tree classifier** has higher **accuracy** 0.8421 than Naive Bayes 0.8341.

**Naive Bayes** has higher **precision** for class 0 (no) and class 1 (yes) compared to Decision Tree.

**Naive Bayes** has higher **recall** for class 1 (yes) compared to Decision Tree, while Decision tree has higher recall for class 0 (no).

**F1-scores** are similar between the two models, with **Naive Bayes** having a slightly higher F1-score for class 1 (yes).

**Overall**, while Decision Tree has a slightly higher accuracy, **Naive Bayes** seems to perform better in others for class 1 (yes).

# **Task 2**
The following is a database of 1698 Hindi Movies from 2005-2017:
https://www.kaggle.com/datasets/rishidamarla/bollywood-movies-dataset

A movie is a hit if revenue > budget, and it is a flop otherwise. The goal is to predict whether a movie will be a hit or flop, given all the other attributes.
Once again, your task is to build two classifiers for this data set: a decision tree and a naïve Bayes classifier. Use a suitable evaluation metric to compare the performance of the classifiers.



In [ ]:
#first step: read data and view a subset from it

df2 = pd.read_csv("/content/Data for repository.csv")
df2.head()

,Movie Name,Release Period,Whether Remake,Whether Franchise,Genre,New Actor,New Director,New Music Director,Lead Star,Director,Music Director,Number of Screens,Revenue(INR),Budget(INR)
0,Golden Boys,Normal,No,No,suspense,Yes,No,No,Jeet Goswami,Ravi Varma,Baba Jagirdar,5,5000000,85000
1,Kaccha Limboo,Holiday,No,No,drama,Yes,No,Yes,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer,75,15000000,825000
2,Not A Love Story,Holiday,No,No,thriller,No,No,No,Mahie Gill,Ram Gopal Verma,Sandeep Chowta,525,75000000,56700000
3,Qaidi Band,Holiday,No,No,drama,Yes,No,No,Aadar Jain,Habib Faisal,Amit Trivedi,800,210000000,4500000
4,Chaatwali,Holiday,No,No,adult,Yes,Yes,Yes,Aadil Khan,Aadil Khan,Babloo Ustad,1,1000000,1075000


In [ ]:
#check the dimension (shape) of the data frame
df2.shape

(1698, 14)

In [ ]:
#check for missing values to handle if exist
df2.isnull().sum()

Movie Name            0
Release Period        0
Whether Remake        0
Whether Franchise     0
Genre                 0
New Actor             0
New Director          0
New Music Director    0
Lead Star             0
Director              0
Music Director        0
Number of Screens     0
Revenue(INR)          0
Budget(INR)           0
dtype: int64

In [ ]:
#check number of unqiue values in each column
df2.nunique()

Movie Name            1695
Release Period           2
Whether Remake           2
Whether Franchise        2
Genre                   14
New Actor                2
New Director             2
New Music Director       2
Lead Star              764
Director              1048
Music Director         630
Number of Screens      147
Revenue(INR)           184
Budget(INR)           1104
dtype: int64

In [ ]:
#check datatype for each column to see what to convert to numerical
df2.dtypes

Movie Name            object
Release Period        object
Whether Remake        object
Whether Franchise     object
Genre                 object
New Actor             object
New Director          object
New Music Director    object
Lead Star             object
Director              object
Music Director        object
Number of Screens      int64
Revenue(INR)           int64
Budget(INR)            int64
dtype: object

In [ ]:
#function to build the target column as described

def Hit_Flop(df):
  if df['Revenue(INR)'] > df['Budget(INR)']:
    return 'Hit'
  else:
    return 'Flop'

df2['Status'] = df2.apply(Hit_Flop,axis=1)
df2.head()

,Movie Name,Release Period,Whether Remake,Whether Franchise,Genre,New Actor,New Director,New Music Director,Lead Star,Director,Music Director,Number of Screens,Revenue(INR),Budget(INR),Status
0,Golden Boys,Normal,No,No,suspense,Yes,No,No,Jeet Goswami,Ravi Varma,Baba Jagirdar,5,5000000,85000,Hit
1,Kaccha Limboo,Holiday,No,No,drama,Yes,No,Yes,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer,75,15000000,825000,Hit
2,Not A Love Story,Holiday,No,No,thriller,No,No,No,Mahie Gill,Ram Gopal Verma,Sandeep Chowta,525,75000000,56700000,Hit
3,Qaidi Band,Holiday,No,No,drama,Yes,No,No,Aadar Jain,Habib Faisal,Amit Trivedi,800,210000000,4500000,Hit
4,Chaatwali,Holiday,No,No,adult,Yes,Yes,Yes,Aadil Khan,Aadil Khan,Babloo Ustad,1,1000000,1075000,Flop


In [ ]:
#replaced (Yes,No,Hit,Flop) with 1,0 convert from categorical to numerical

df2.replace({'Yes': 1, 'No': 0,'Hit': 1, 'Flop': 0}, inplace=True)
df2.head()

,Movie Name,Release Period,Whether Remake,Whether Franchise,Genre,New Actor,New Director,New Music Director,Lead Star,Director,Music Director,Number of Screens,Revenue(INR),Budget(INR),Status
0,Golden Boys,Normal,0,0,suspense,1,0,0,Jeet Goswami,Ravi Varma,Baba Jagirdar,5,5000000,85000,1
1,Kaccha Limboo,Holiday,0,0,drama,1,0,1,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer,75,15000000,825000,1
2,Not A Love Story,Holiday,0,0,thriller,0,0,0,Mahie Gill,Ram Gopal Verma,Sandeep Chowta,525,75000000,56700000,1
3,Qaidi Band,Holiday,0,0,drama,1,0,0,Aadar Jain,Habib Faisal,Amit Trivedi,800,210000000,4500000,1
4,Chaatwali,Holiday,0,0,adult,1,1,1,Aadil Khan,Aadil Khan,Babloo Ustad,1,1000000,1075000,0


In [ ]:
#check unique values of Genre
df2['Genre'].unique()

array(['suspense', 'drama', 'thriller', 'adult', 'comedy', 'action',
       'love_story', 'rom__com', 'horror', 'fantasy', 'masala',
       'mythological', 'animation', 'documentary'], dtype=object)

In [ ]:
df2['Release Period'].unique()

array(['Normal', 'Holiday'], dtype=object)

In [ ]:
#converting the values to numerical
df2['Release Period'] = df2['Release Period'].replace({'Holiday':1, 'Normal':0})
df2.head()

,Movie Name,Release Period,Whether Remake,Whether Franchise,Genre,New Actor,New Director,New Music Director,Lead Star,Director,Music Director,Number of Screens,Revenue(INR),Budget(INR),Status
0,Golden Boys,0,0,0,suspense,1,0,0,Jeet Goswami,Ravi Varma,Baba Jagirdar,5,5000000,85000,1
1,Kaccha Limboo,1,0,0,drama,1,0,1,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer,75,15000000,825000,1
2,Not A Love Story,1,0,0,thriller,0,0,0,Mahie Gill,Ram Gopal Verma,Sandeep Chowta,525,75000000,56700000,1
3,Qaidi Band,1,0,0,drama,1,0,0,Aadar Jain,Habib Faisal,Amit Trivedi,800,210000000,4500000,1
4,Chaatwali,1,0,0,adult,1,1,1,Aadil Khan,Aadil Khan,Babloo Ustad,1,1000000,1075000,0


In [ ]:
#convert (Genre) to numerical using one-hot encoder, because there's no order between them and it doesn't have alot of unique values
df2 = pd.get_dummies(df2, columns = ['Genre'])
df2.head()

,Movie Name,Release Period,Whether Remake,Whether Franchise,New Actor,New Director,New Music Director,Lead Star,Director,Music Director,...,Genre_documentary,Genre_drama,Genre_fantasy,Genre_horror,Genre_love_story,Genre_masala,Genre_mythological,Genre_rom__com,Genre_suspense,Genre_thriller
0,Golden Boys,0,0,0,1,0,0,Jeet Goswami,Ravi Varma,Baba Jagirdar,...,0,0,0,0,0,0,0,0,1,0
1,Kaccha Limboo,1,0,0,1,0,1,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer,...,0,1,0,0,0,0,0,0,0,0
2,Not A Love Story,1,0,0,0,0,0,Mahie Gill,Ram Gopal Verma,Sandeep Chowta,...,0,0,0,0,0,0,0,0,0,1
3,Qaidi Band,1,0,0,1,0,0,Aadar Jain,Habib Faisal,Amit Trivedi,...,0,1,0,0,0,0,0,0,0,0
4,Chaatwali,1,0,0,1,1,1,Aadil Khan,Aadil Khan,Babloo Ustad,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#splitting the data to X and y, for X i dropped the target (status), other features because they have a lot of unique values and there's no order
#between them so using one-hot encoder or label encoder won't be a good choice, i removed them because i don't think they will affect the result

X = df2.drop(columns=['Status','Movie Name','Director', 'Music Director','Lead Star'])
y = df2['Status']

In [ ]:
print(X.columns)
X.head()

Index(['Release Period', 'Whether Remake', 'Whether Franchise', 'New Actor',
       'New Director', 'New Music Director', 'Number of Screens',
       'Revenue(INR)', 'Budget(INR)', 'Genre_action', 'Genre_adult',
       'Genre_animation', 'Genre_comedy', 'Genre_documentary', 'Genre_drama',
       'Genre_fantasy', 'Genre_horror', 'Genre_love_story', 'Genre_masala',
       'Genre_mythological', 'Genre_rom__com', 'Genre_suspense',
       'Genre_thriller'],
      dtype='object')


,Release Period,Whether Remake,Whether Franchise,New Actor,New Director,New Music Director,Number of Screens,Revenue(INR),Budget(INR),Genre_action,...,Genre_documentary,Genre_drama,Genre_fantasy,Genre_horror,Genre_love_story,Genre_masala,Genre_mythological,Genre_rom__com,Genre_suspense,Genre_thriller
0,0,0,0,1,0,0,5,5000000,85000,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,1,0,1,75,15000000,825000,0,...,0,1,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,525,75000000,56700000,0,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,1,0,0,800,210000000,4500000,0,...,0,1,0,0,0,0,0,0,0,0
4,1,0,0,1,1,1,1,1000000,1075000,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
#split the data to train 70% and test 30%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

**Decision Tree Classifier**

In [ ]:
decisionTree = DecisionTreeClassifier()
decisionTree.fit(X_train,y_train)

y_predict = decisionTree.predict(X_test)
accuracy = accuracy_score(y_test, y_predict)
report = classification_report(y_test, y_predict)

print("Accuracy of decision Tree:", accuracy)
print('\n\t\t\t -REPORT- \n')
print(report)

Accuracy of decision Tree: 0.9745098039215686

			 -REPORT- 

              precision    recall  f1-score   support

           0       0.92      0.99      0.95       133
           1       1.00      0.97      0.98       377

    accuracy                           0.97       510
   macro avg       0.96      0.98      0.97       510
weighted avg       0.98      0.97      0.97       510



In [ ]:
# 1- build (Decision Tree Classifier)
# 2- train the classifier (using fit)
# 3- apply what it learned during fitting to predict the class (using predict)
# 4- evaluate the performance of the classifier
# 5- report that includes precision, recall, F1-score, and support for evaluating the performance of the classifier

**Naïve Bayes Classifier**

In [ ]:
naiveBayes = GaussianNB()
naiveBayes.fit(X_train,y_train)

y_predict2 = naiveBayes.predict(X_test)
accuracy2 = accuracy_score(y_test, y_predict2)
report2 = classification_report(y_test, y_predict2)

print("Accuracy of naive Bayes:", accuracy2)
print('\n\t\t\t -REPORT- \n')
print(report2)

Accuracy of naive Bayes: 0.8862745098039215

			 -REPORT- 

              precision    recall  f1-score   support

           0       0.86      0.68      0.76       133
           1       0.89      0.96      0.93       377

    accuracy                           0.89       510
   macro avg       0.88      0.82      0.84       510
weighted avg       0.88      0.89      0.88       510



In [ ]:
# 1- build (Naïve Bayes Classifier)
# 2- train the classifier (using fit)
# 3- apply what it learned during fitting to predict the class (using predict)
# 4- evaluate the performance of the classifier
# 5- report that includes precision, recall, F1-score, and support for evaluating the performance of the classifier

**Decision Tree classifier** has higher **accuracy** 0.9745, this means that the performance of it is better than Naive Bayes 0.88627.

**Precision:** For both classes, the **Decision Tree classifier** has higher precision compared to Naive Bayes.

**Recall:** The **Decision Tree classifier** shows better recall score for both classes compared to Naive Bayes.

**F1-score:** The F1-scores for both classes are higher for the **Decision Tree classifier**.

so overall, **Decision Tree classifier** has better performance in correctly classifying instances and capturing true positives for both classes compared to Naive Bayes.

**time taken:** 7-8 hours (most of the time on task1)

**space required:** 2.5 GB